# Authentication Mechanism 2

CEDA data - via ceda account, http using token


In [2]:
%load_ext autoreload
%autoreload 2

curl --location --request POST 'https://services.ceda.ac.uk/api/token/create/' --header "Authorization: Basic $(echo -n "...:..." | base64)"


In [8]:
import getpass
import json
import requests

from base64 import b64encode
from pathlib import Path
from urllib.parse import urlparse


In [2]:
CEDA_USERNAME = input("Enter DAFNI username: ")
CEDA_PASSWORD = getpass.getpass("Enter DAFNI password: ")

if not CEDA_USERNAME or not CEDA_PASSWORD:
    raise ValueError("CEDA_USERNAME and/or CEDA_PASSWORD were not provided.")

In [3]:
url = "https://services.ceda.ac.uk/api/token/create/"

username = CEDA_USERNAME
password = CEDA_PASSWORD
token = b64encode(f"{username}:{password}".encode("utf-8")).decode("ascii")
headers = {
    "Authorization": f"Basic {token}",
}

response = requests.request("POST", url, headers=headers)

# If successful, this will return a JSON response containing the token
response_data = json.loads(response.text)
print(response.text)
if response.status_code == 200:
    token = response_data["access_token"]

{"access_token":"eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICI4ZjhmaUpyaUtDY3hmaHhzdU5vazVEekdJdFZ4amhhTWNJa05ZX2U4MnhJIn0.eyJleHAiOjE3NjkxNjk5MjEsImlhdCI6MTc2ODkxMDcyMSwianRpIjoiZWFlNWI5OTctMjdhYy00NWE1LWI5ZDItMDY5Y2EwNWE4OTUxIiwiaXNzIjoiaHR0cHM6Ly9hY2NvdW50cy5jZWRhLmFjLnVrL3JlYWxtcy9jZWRhIiwic3ViIjoiZjQ3OGIwZGMtY2UzNC00ODU4LTk2M2UtN2Q2Zjk3Yzc5MDdlIiwidHlwIjoiQmVhcmVyIiwiYXpwIjoic2VydmljZXMtcG9ydGFsLWNlZGEtYWMtdWsiLCJzZXNzaW9uX3N0YXRlIjoiYWY0N2M3NGUtYzZmNC00ODlmLTljMWQtNWM4MmEwZTY2ZmI3IiwiYWNyIjoiMSIsInNjb3BlIjoiZW1haWwgb3BlbmlkIHByb2ZpbGUgZ3JvdXBfbWVtYmVyc2hpcCIsInNpZCI6ImFmNDdjNzRlLWM2ZjQtNDg5Zi05YzFkLTVjODJhMGU2NmZiNyIsImVtYWlsX3ZlcmlmaWVkIjp0cnVlLCJuYW1lIjoiU2FpZnVsIEtoYW4iLCJwcmVmZXJyZWRfdXNlcm5hbWUiOiJzYWlmdWxraGFuIiwiZ2l2ZW5fbmFtZSI6IlNhaWZ1bCIsImZhbWlseV9uYW1lIjoiS2hhbiIsImVtYWlsIjoic2FpZnVsLmtoYW5Ac3RmYy5hYy51ayJ9.ORRdT0iF3QW5utDiqFWZovHAqMgl7MEfUCuhPQD5aW5B38W4IV_rK6qkHK8oESYvJ3XkbIE2ZRxVVZe21y95zBSjrN_xX82cQ9jyyDMX2mj0w_S3cpdAGZ63XsTJFbUgLKjjjOftOdWCisr2tYqC8xLcXu3RmT

In [10]:
data_url = (
    "https://dap.ceda.ac.uk/badc/csip/data/salford-radiometer-1/2005/06/salford-radiometer-1_faccombe_20050624_iwv.nc"
)

output_dir = Path("../data/download")
output_dir.mkdir(parents=True, exist_ok=True)

filename = Path(urlparse(data_url).path).name
output_file = output_dir / filename

with requests.get(data_url, headers=headers, stream=True, timeout=30) as r:
    r.raise_for_status()  # <-- fail fast on 403/404/etc.

    with open(output_file, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

print(f"Downloaded → {output_file}")

requests.get(data_url, headers={"Authorization": f"Bearer {token}"}, stream=True, timeout=30)

Downloaded → ../data/download/salford-radiometer-1_faccombe_20050624_iwv.nc


<Response [200]>